# Phase 1: Watching optimizers move on low-dimensional surfaces

In this notebook we optimize a single 2-D point over a few classic loss surfaces and **watch the path** each optimizer takes. The goal is intuition, not realism:

- See how **SGD**, **momentum**, **AdaGrad**, and **Adam** differ.
- Build a feel for convergence speed, stability, and sensitivity to the learning rate.

The optimizers here are written from scratch in [`workshoplib/optimizers.py`](../workshoplib/optimizers.py) so you can read the exact update rules. (MuON is introduced later, in Phase 2, where the network has real weight *matrices* for it to act on.)

In [ ]:
from pathlib import Path
import os
import sys

# Make sure we run from the project root so 'import workshoplib' works.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    os.chdir(ROOT.parent)
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Working directory:", ROOT)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from workshoplib.objectives import OBJECTIVES, get_objective
from workshoplib.optimizers import run_descent
from workshoplib import viz

OPTIMIZERS = ["sgd", "momentum", "adagrad", "adam"]
print("Available objectives:", list(OBJECTIVES))

## A loss surface and a gradient step

Each objective is a function that maps a 2-D point `(x, y)` to a single number (the *loss*). We can picture it as a landscape: the height is the loss, and optimization is the process of walking downhill.

Every optimizer here does the same basic thing each step: compute the gradient (the steepest-uphill direction) and move in the opposite direction. They differ in **how** they use that gradient (raw, smoothed with momentum, or rescaled per-coordinate).

The helper below runs all four optimizers from the same starting point and shows two views: the **path on the contour map** and the **loss vs. step** curve.

In [ ]:
def compare(key, n_steps=80):
    """Run every optimizer on one objective and show both diagnostic plots."""
    objective = get_objective(key)
    trajectories = [run_descent(objective, name, n_steps=n_steps) for name in OPTIMIZERS]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    viz.plot_trajectory_on_contour(objective, trajectories, ax=axes[0])
    viz.plot_loss_curves(trajectories, ax=axes[1])
    fig.tight_layout()
    plt.show()
    return trajectories

## 1. Quadratic bowl (the easy case)

`f = x^2 + y^2`. A symmetric bowl with its minimum at the origin. Here almost everything works: it is a good sanity check and baseline before we make life harder.

In [ ]:
_ = compare("quadratic_bowl", n_steps=80)

**What to notice:** all four methods head straight for the center. Differences are small because the surface is perfectly conditioned (equally curved in every direction).

## 2. Ill-conditioned quadratic (the stretched bowl)

`f = x^2 + 25 y^2`. Now the bowl is 25x steeper in `y` than in `x`. A single learning rate is a compromise: large enough to make progress in the shallow `x` direction tends to overshoot and oscillate in the steep `y` direction.

In [ ]:
_ = compare("ill_conditioned", n_steps=80)

**What to notice:** plain SGD zig-zags down the steep walls and crawls along the shallow valley floor. Momentum smooths the zig-zag; AdaGrad and Adam rescale each coordinate's step and reach the center far more directly. This per-coordinate scaling is the whole point of adaptive methods.

## 3. Rosenbrock (the curved valley)

`f = (1 - x)^2 + 100 (y - x^2)^2`. The famous "banana". The minimum at `(1, 1)` sits at the bottom of a long, curved, narrow valley. It is easy to drop into the valley but hard to follow it to the bottom, so we give it more steps.

In [ ]:
_ = compare("rosenbrock", n_steps=300)

**What to notice:** plain SGD and AdaGrad stall partway along the valley. Momentum and Adam carry speed around the bend and make much more progress toward `(1, 1)`. This is where momentum really earns its keep.

## 4. Saddle point

`f = x^2 - y^2`. There is **no minimum**: the origin is a saddle (a minimum along `x`, a maximum along `y`). Optimizers should slide down along `y` and away. This illustrates that not every flat-looking point is a good place to stop.

In [ ]:
_ = compare("saddle", n_steps=80)

**What to notice:** the loss keeps decreasing (it can go negative here) as the optimizers escape along the downhill `y` direction. Methods that build up speed or rescale steps leave the saddle's flat ridge faster than plain SGD.

## 5. Beale (optional, harder)

A multi-feature surface with broad flat regions and sharp valleys, minimum at `(3, 0.5)`. A good stress test for keen students.

In [ ]:
_ = compare("beale", n_steps=300)

**What to notice:** the flat regions starve plain SGD of gradient, so it barely moves; momentum and Adam make far more progress toward the minimum.

## 6. Learning-rate sensitivity

The learning rate is the single most important knob. Too small and training crawls; too large and it oscillates or diverges. Below we run plain **SGD** on the ill-conditioned quadratic at several learning rates and compare the loss curves.

The steep direction has curvature 50, so SGD is only stable for learning rates below about `2 / 50 = 0.04`. Watch what happens as we cross that threshold.

In [ ]:
objective = get_objective("ill_conditioned")
learning_rates = [0.005, 0.02, 0.038, 0.045]

fig, ax = plt.subplots(figsize=(7, 5))
for lr in learning_rates:
    traj = run_descent(objective, "sgd", lr=lr, n_steps=60)
    ax.plot(traj.losses.numpy(), label=f"lr = {lr}")

ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title("SGD learning-rate sensitivity (ill-conditioned quadratic)")
ax.legend()
plt.show()

**What to notice:** tiny rates converge slowly; a well-chosen rate converges fast; a rate past the stability limit makes the loss grow instead of shrink.

## Try it yourself

- Change the starting point: `run_descent(get_objective("rosenbrock"), "adam", x0=(1.5, -0.5), n_steps=300)`.
- Override learning rates per optimizer and see when each diverges.
- Add your own 2-D objective to [`workshoplib/objectives.py`](../workshoplib/objectives.py) and rerun `compare("your_key")`.
- Visualize a surface in 3-D: `viz.plot_surface_3d(get_objective("rosenbrock")); plt.show()`.